# 1. MLP and Input-Output Convex Neural Networks

# 1.2 MLP implementation from scratch

In [1]:
%pip install numpy datasets matplotlib wandb

  Using cached pyyaml-6.0.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 55.2 kB/s  0:04:51 eta 0:00:040:00:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 40.1 kB/s  0:00:15 37.3 kB/s eta 0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 47.9 kB/s  0:00:167.7 kB/s eta 0:00:02:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 53.1 kB/s  0:01:20 eta 0:00:010:06:11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 60.7 kB/s  0:02:47a 0:00:030:00:07:27
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 57.6 kB/s  0:05:56 eta 0:00:030:00:12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 40.9 kB/s  0:00:48 40.2 kB/s eta 0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 47.8 kB/s  0:00:396.7 kB/s eta 0:00:02:08
Using cached cycler-0.12.1-py3-none-any

In [1]:
import numpy as np

ROLL_NUMBER = 2026701036
SEED = ROLL_NUMBER % 100000
RUN = 0
print("Seed for question 1:", SEED)

Seed for question 1: 1036


In [8]:
from datasets import load_dataset

ds = load_dataset("uoft-cs/cifar10")

X = np.empty((60_000,32,32,3),dtype=np.uint8)
y = np.empty(60_000, dtype=np.int64)

for i, img in enumerate(ds["train"]["img"]):
    X[i] = np.asarray(img, dtype=np.uint8)
for i, img in enumerate(ds["test"]["img"]):
    X[50_000 + i] = np.asarray(img, dtype=np.uint8)

y[:50_000] = np.asarray(ds["train"]["label"], dtype=np.int64)
y[50_000:] = np.asarray(ds["test"]["label"], dtype=np.int64)

print("X:", X.shape,X.dtype)
print("y:", y.shape,y.dtype)
print("per-class counts:", np.bincount(y, minlength=10))

X: (60000, 32, 32, 3) uint8
y: (60000,) int64
per-class counts: [6000 6000 6000 6000 6000 6000 6000 6000 6000 6000]


In [9]:
X = X.reshape(60_000,32*32*3)

def stratified_split(y, rng, n_train, n_val, n_test):
    train_idx, val_idx, test_idx = [], [], []
    for c in np.unique(y):
        idx = np.flatnonzero(y == c)
        idx = rng.permutation(idx)
        assert len(idx) >= n_train+n_val+n_test
        train_idx.append(idx[:n_train])
        val_idx.append(idx[n_train:n_train+n_val])
        test_idx.append(idx[n_train+n_val:n_train+n_val+n_test])
    return (np.concatenate(train_idx),np.concatenate(val_idx),np.concatenate(test_idx))

rng_split = np.random.default_rng(SEED+100*RUN+0)
train_idx,val_idx,test_idx = stratified_split(y,rng_split,3600,1200,1200)

print("train counts:", np.bincount(y[train_idx],minlength=10))
print("val counts:", np.bincount(y[val_idx],minlength=10))
print("test counts:", np.bincount(y[test_idx],minlength=10))

train counts: [3600 3600 3600 3600 3600 3600 3600 3600 3600 3600]
val counts: [1200 1200 1200 1200 1200 1200 1200 1200 1200 1200]
test counts: [1200 1200 1200 1200 1200 1200 1200 1200 1200 1200]


In [11]:
X_train = X[train_idx].astype(np.float64)
X_val = X[val_idx].astype(np.float64)
X_test = X[test_idx].astype(np.float64)

X_train /= 255.0
X_val /= 255.0
X_test /= 255.0

y_train = y[train_idx].copy()
y_val = y[val_idx].copy()
y_test = y[test_idx].copy()

train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)

train_std[train_std ==0.0] = 1.0

X_train -= train_mean
X_train /= train_std
X_val -= train_mean
X_val /= train_std
X_test -= train_mean
X_test /= train_std

print("train mean first 3:", X_train.mean(axis=0)[:3])
print("train std first 3:", X_train.std(axis=0)[:3])
print("test mean first 3:", X_test.mean(axis=0)[:3])
print("test std first 3:", X_test.std(axis=0)[:3])

print("X_train:", X_train.shape, X_train.dtype)
print("X_val:", X_val.shape, X_val.dtype)
print("X_test:", X_test.shape, X_test.dtype)

train mean first 3: [1.40387597e-14 4.61716899e-14 4.00795570e-14]
train std first 3: [1. 1. 1.]
test mean first 3: [ 0.00231415  0.00064378 -0.00472698]
test std first 3: [1.00449637 1.00379161 1.00436622]
X_train: (36000, 3072) float64
X_val: (12000, 3072) float64
X_test: (12000, 3072) float64


the test mean and std of the above results are not exactly 0 and 1 because the standardisation statistics were computed on the training split only, the test split is different sample and therefore has slightly differnet statis

In [13]:
print(y_train.dtype, y_train.shape, np.unique(y_train))
print(y_val.dtype, y_val.shape, np.unique(y_val))
print(y_test.dtype, y_test.shape, np.unique(y_test))

int64 (36000,) [0 1 2 3 4 5 6 7 8 9]
int64 (12000,) [0 1 2 3 4 5 6 7 8 9]
int64 (12000,) [0 1 2 3 4 5 6 7 8 9]
